# Weekly C Progress — Streaming Sensor Analyzer

## Goal

This notebook documents my progress on a C programming exercise focused on:

- streaming file input
- parsing delimited text
- validating numeric input
- using `struct`
- tracking per-device statistics
- separating responsibilities across functions
- handling malformed input safely
- thinking about memory ownership and scalability

This is a **progress notebook**, not a polished final submission. It shows how my design evolved and what I learned while building the solution.

## Original Task

A factory produces records like:

```text
2026-09-13T08:14:31;MOTOR-03;72.4;OK
2026-09-13T08:14:32;PUMP-01;91.7;WARNING
2026-09-13T08:14:33;MOTOR-03;105.2;CRITICAL
2026-09-13T08:14:34;VALVE-07;INVALID;OK
BROKEN LINE
2026-09-13T08:14:36;PUMP-01;94.3;WARNING
```

The program must eventually produce per-device statistics such as:

```text
Device: MOTOR-03
Valid readings: 2
Average: 88.80
Minimum: 72.40
Maximum: 105.20
Warnings: 0
Critical: 1
```

and finally:

```text
Malformed records: 2
```

Important constraints include:

- unknown number of records/devices
- at least one `struct`
- no global mutable state
- malformed input must not crash the program
- parsing errors must be distinguished from file/allocation errors
- responsibilities should be separated instead of putting everything in `main()`
- the design should scale conceptually to a 40 GB input file

# 1. My Initial Approach

My first attempt tried to manually read individual characters by position and split the timestamp, model, value, and state using index arithmetic.

Example of the original direction:

```c
for (i = 0; i < 10; i++) {
    char date[11] = "";
    date[i] = input[i];
}
```

This led to several problems:

- `input` had not yet been filled from the file.
- temporary arrays such as `date` were declared inside loops and disappeared every iteration.
- I hard-coded character positions even though the input already contained delimiters (`;`).
- I was trying to solve parsing, storage, and statistics all at once.

### Main lesson

The input format already defines fields. I should think in terms of:

```text
record
→ field 1
→ field 2
→ field 3
→ field 4
```

rather than individual character positions.

# 2. Moving to Line-Based Parsing

I changed the program to use:

```c
fgets(...)
```

to read one complete line at a time.

Then I moved to `sscanf()` with scansets:

```c
sscanf(
    input,
    "%19[^;];%19[^;];%19[^;];%19[^\n]",
    ...
)
```

This was a major improvement because the record is naturally semicolon-separated.

I also realized that the timestamp should be treated as **one field**:

```text
2026-09-13T08:14:31
```

not as separate date and time fields during parsing.

# 3. Temporary Parsed Data

I introduced a struct to hold the current parsed record:

```c
typedef struct {
    char dateandTime[20];
    char model[20];
    char valuetext[20];
    char state[20];
} Data;
```

The important design idea is that this struct represents **only the current input line**.

It should not become a giant array containing every reading in the file.

The current record can be parsed, validated, used to update statistics, and then overwritten by the next call to `fgets()`.

# 4. Numeric Validation

A major problem in the input is that the reading field may contain:

```text
72.4
105.2
INVALID
```

So simply calling `atof()` is not enough because I need to know whether conversion actually succeeded.

I switched to:

```c
strtof()
```

with an end pointer:

```c
char *end;
float value = strtof(data.valuetext, &end);
```

Validation logic:

```c
if (end == data.valuetext || *end != '\0') {
    // invalid number
}
```

### Why this works

- `end == data.valuetext` means no numeric characters were parsed.
- `*end != '\0'` means conversion stopped before the end of the string.
- otherwise the entire string was successfully parsed as a float.

Examples:

```text
"72.4"     → valid
"INVALID"  → invalid
"72.4abc"  → invalid
```

# 5. Important Design Shift: Do Not Store Every Reading

At first I had:

```c
Data list[MAX_DATA];
```

and planned to save every record.

This is not a good streaming design.

For the required output, I do not need every raw reading after I have used it.

For example, after seeing:

```text
MOTOR-03;72.4;OK
MOTOR-03;105.2;CRITICAL
```

I only need to retain:

```text
count
sum
minimum
maximum
warning count
critical count
```

This is the key idea behind the 40 GB extension as well.

# 6. Per-Device Statistics

I introduced a second struct for accumulated statistics:

```c
typedef struct {
    char Device[20];
    int Valid_readings;
    float Sum;
    float Average;
    float Minimum;
    float Maximum;
    int WARNING;
    int CRITICAL;
} output;
```

The intended meaning is now:

```text
stats[0] → statistics for one unique device
stats[1] → statistics for another unique device
...
```

This is very different from:

```text
one element per input record
```

The array should represent **unique devices**, not all readings.

# 7. Finding or Creating a Device

The next challenge was avoiding duplicate entries such as:

```text
stats[0] = MOTOR-03
stats[1] = PUMP-01
stats[2] = MOTOR-03   ← wrong
```

The solution is to search existing devices first.

Conceptually:

```text
new valid reading arrives
        ↓
search existing stats entries
        ↓
device found?
   yes → return existing index
   no  → create new stats entry and return new index
```

The important state variable is:

```c
device_count
```

which means:

> number of unique device statistics entries currently stored

# 8. Separating Responsibilities

A major improvement was separating:

```text
find/create device
```

from:

```text
update statistics
```

The intended structure is now:

```text
main()
    ↓
read line
    ↓
parse
    ↓
validate
    ↓
find_or_create_device()
    ↓
update_stats()
```

This is much easier to reason about than putting all behavior into `main()`.

## `find_or_create_device()`

This function should only:

- search for an existing device
- return its index if found
- otherwise create a new empty statistics entry
- increment `device_count`
- return the new index

It should **not** modify sums, averages, minimums, maximums, or status counters.

## `update_stats()`

This function handles one valid reading.

A useful pattern is:

```c
if (stat->Valid_readings == 0) {
    stat->Minimum = value;
    stat->Maximum = value;
} else {
    if (value < stat->Minimum) {
        stat->Minimum = value;
    }

    if (value > stat->Maximum) {
        stat->Maximum = value;
    }
}
```

Then status counters are updated:

```c
if (strcmp(state, "WARNING") == 0) {
    stat->WARNING++;
} else if (strcmp(state, "CRITICAL") == 0) {
    stat->CRITICAL++;
}
```

Finally:

```c
stat->Valid_readings++;
stat->Sum += value;
stat->Average = stat->Sum / stat->Valid_readings;
```

This fixed an important bug where I was incrementing `Valid_readings` before checking whether it was the first reading.

# 9. Malformed Records

I learned that malformed records should **not** create or modify device statistics.

Examples:

```text
2026-09-13T08:14:34;VALVE-07;INVALID;OK
BROKEN LINE
```

Both should increase:

```c
malformed++;
```

but should not call:

```c
find_or_create_device(...)
update_stats(...)
```

Using fake values such as:

```c
0.0
```

for malformed records would corrupt the statistics.

Likewise, using sentinel values such as:

```c
-1.0
```

is dangerous because `-1.0` may be a legitimate sensor reading.

# 10. Current Processing Pipeline

The current target flow is:

```text
while (fgets(...)) {

    parse line

    if field parsing failed:
        malformed++
        continue

    convert numeric text

    if numeric conversion failed:
        malformed++
        continue

    find/create device

    if device storage failed:
        handle storage error

    update device statistics
}
```

This is significantly cleaner than my initial character-by-character parser.

# 11. Current Work-in-Progress Code

The following represents the current stage of development. It is intentionally not presented as a finished submission.

In [ ]:
#include <string.h>
#include <stdio.h>
#include <stdbool.h>
#include <stdlib.h>

#define MAX_DATA 10000

typedef struct {
    char dateandTime[20];
    char model[20];
    char valuetext[20];
    char state[20];
} Data;

typedef struct {
    char Device[20];
    int Valid_readings;
    float Sum;
    float Average;
    float Minimum;
    float Maximum;
    int WARNING;
    int CRITICAL;
} output;

void update_stats(output *stat, float value, const char *state) {
    if (stat->Valid_readings == 0) {
        stat->Minimum = value;
        stat->Maximum = value;
    } else {
        if (value < stat->Minimum) {
            stat->Minimum = value;
        }

        if (value > stat->Maximum) {
            stat->Maximum = value;
        }
    }

    if (strcmp(state, "WARNING") == 0) {
        stat->WARNING++;
    } else if (strcmp(state, "CRITICAL") == 0) {
        stat->CRITICAL++;
    }

    stat->Valid_readings++;
    stat->Sum += value;
    stat->Average = stat->Sum / stat->Valid_readings;
}

int find_or_create_device(output stats[],
                          int *device_count,
                          const char *model) {
    if (device_count == NULL || model == NULL) {
        return -1;
    }

    for (int i = 0; i < *device_count; i++) {
        if (strcmp(stats[i].Device, model) == 0) {
            return i;
        }
    }

    strcpy(stats[*device_count].Device, model);
    stats[*device_count].Valid_readings = 0;
    stats[*device_count].Sum = 0.0f;
    stats[*device_count].Average = 0.0f;
    stats[*device_count].WARNING = 0;
    stats[*device_count].CRITICAL = 0;

    (*device_count)++;

    return *device_count - 1;
}

# 12. Current Main-Loop Logic

The current direction for `main()` is:

```c
while (fgets(input, sizeof input, file) != NULL) {

    int fields = sscanf(
        input,
        "%19[^;];%19[^;];%19[^;];%19[^\n]",
        data.dateandTime,
        data.model,
        data.valuetext,
        data.state
    );

    if (fields != 4) {
        malformed++;
        continue;
    }

    char *end;
    float value = strtof(data.valuetext, &end);

    if (end == data.valuetext || *end != '\0') {
        malformed++;
        continue;
    }

    int index =
        find_or_create_device(stats, &device_count, data.model);

    if (index == -1) {
        /* storage / capacity error */
    }

    update_stats(&stats[index], value, data.state);
}
```

This is the first version where the control flow matches the intended architecture.

# 13. Problems Still Remaining

The implementation is not finished yet.

Current issues to solve:

- `MAX_DATA` is still a fixed device limit.
- `find_or_create_device()` needs a capacity check before writing `stats[*device_count]`.
- the assignment says the number of devices is unknown, so eventually dynamic allocation would be more appropriate.
- valid states should probably be restricted to `OK`, `WARNING`, and `CRITICAL`.
- output/reporting still needs to be implemented.
- malformed line handling should be tested carefully.
- file name should eventually come from `argv`, because the required program is:

```text
sensorstat readings.txt
```

- `Average` does not strictly need to be stored because it can be calculated as:

```c
Sum / Valid_readings
```

during reporting.

# 14. Three Malformed Test Cases

These are useful deliberate failure cases:

### Invalid numeric value

```text
2026-09-13T08:14:34;VALVE-07;INVALID;OK
```

Expected behavior:

```text
malformed++
```

No statistics update.

### Missing fields

```text
BROKEN LINE
```

Expected behavior:

```text
sscanf(...) != 4
malformed++
```

### Partially numeric value

```text
2026-09-13T08:15:00;MOTOR-03;72.4abc;OK
```

`strtof()` can parse `72.4`, but `end` does not point to `' '`.

Expected behavior:

```text
malformed++
```

# 15. Memory / 40 GB Design Insight

A function such as:

```c
Reading *read_all_readings(FILE *file, size_t *count);
```

would require storing every individual reading.

If there are `R` readings, the memory complexity is approximately:

```text
O(R)
```

For a 40 GB file, retaining all records would be impractical.

The streaming design instead keeps:

- one temporary parsed record
- one accumulated statistics entry per unique device

If there are `D` unique devices, memory usage is approximately:

```text
O(D)
```

The number of input records can grow enormously without requiring memory proportional to the file size.

This is one of the most important architectural lessons from the exercise.

# 16. Ownership Notes So Far

The current implementation uses stack-allocated structures, so there is not yet dynamic-memory ownership to manage.

Current ownership is simple:

```text
main()
owns:
    input buffer
    temporary Data record
    stats array
    FILE *
```

The file is opened with:

```c
fopen(...)
```

and released with:

```c
fclose(...)
```

If the statistics array becomes dynamically allocated later, ownership must become explicit:

```text
main / statistics container
    owns allocated stats memory
    must free it exactly once
```

That future step will be important for meeting the assignment's allocation/cleanup requirement properly.

# 17. What I Learned This Week

The biggest improvements were not syntax changes. They were changes in how I approached the problem.

### Parsing
I stopped thinking in individual character positions and started thinking in delimited fields.

### Validation
I learned the difference between conversion and validated conversion using `strtof()` and an end pointer.

### Data modeling
I separated:

```text
one temporary reading
```

from:

```text
persistent per-device statistics
```

### Streaming
I realized that I do not need to store every record to calculate count, sum, min, max, warning count, or critical count.

### Function design
I separated:

```text
find/create device
```

from:

```text
update statistics
```

### Error handling
Malformed input should be counted and skipped, not transformed into fake numeric values.

This was the main progression from the first attempt to the current design.

# 18. Next Steps

The next development tasks are:

1. add capacity protection to `find_or_create_device()`
2. validate sensor states
3. implement reporting
4. use `argv[1]` instead of hard-coded `read.txt`
5. remove unused variables and simplify the structs
6. test with the required malformed cases
7. compile with:

```text
gcc -std=c17 -Wall -Wextra -Wpedantic -Wconversion -g
```

8. test with sanitizers where available:

```text
-fsanitize=address,undefined
```

9. replace the fixed statistics array with dynamic storage if completing the assignment strictly
10. document ownership and cleanup paths